# 草書 Forced Alignment Experiment

Uses Qwen2.5-VL-3B to locate characters in a calligraphy column given the known 釋文 sequence.

**Run cells top to bottom. Make sure you have a T4 GPU runtime enabled.**

In [ ]:
# Cell 1: Install dependencies
!pip install -q sympy==1.13.3
!pip install -q -U transformers
!pip install -q qwen-vl-utils Pillow

In [ ]:
# Cell 2: Authenticate with HuggingFace
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HUGGING_FACE_API')

from huggingface_hub import login
login(os.environ['HF_TOKEN'])
print('Authenticated')

In [ ]:
# Cell 3: Load Qwen2.5-VL-3B
# Downloads ~6GB — takes 3-5 minutes on first run
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-3B-Instruct',
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
print('Model loaded on:', next(model.parameters()).device)

In [ ]:
# Cell 4: Upload your files
# Upload col_experiment.jpg and col_boxes.json from your Downloads folder
from google.colab import files
uploaded = files.upload()

In [ ]:
# Cell 5: Run forced alignment
import json
from PIL import Image
from qwen_vl_utils import process_vision_info

img = Image.open('col_experiment.jpg')
boxes = json.load(open('col_boxes.json'))
chars = [b['char'] for b in boxes]
char_str = ''.join(chars)
n = len(chars)

print(f'Image size: {img.size}')
print(f'Characters ({n}): {char_str}')

messages = [{
    'role': 'user',
    'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': (
            f'This is a column from a Chinese 草書 (cursive calligraphy) scroll, '
            f'read top to bottom. It contains exactly {n} characters in this order: {char_str}\n\n'
            f'Locate each character and output ONLY a JSON array with no other text:\n'
            f'[{{"char":"X","x":10,"y":5,"w":80,"h":15}},...] '
            f'where x,y,w,h are percentages of image dimensions (0-100). '
            f'Output exactly {n} entries in the same order as the character list.'
        )},
    ],
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text], images=image_inputs, videos=video_inputs,
    padding=True, return_tensors='pt'
).to('cuda')

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=512)

trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
output = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
print('\nModel output:')
print(output)

In [ ]:
# Cell 6: Score predictions against ground truth with IoU
# IoU (Intersection over Union) measures how well the predicted box
# overlaps the ground truth box. 1.0 = perfect, 0.0 = no overlap.
# Threshold: >=0.5 good, 0.3-0.5 partial, <0.3 miss

start = output.find('[')
end = output.rfind(']') + 1
if start == -1:
    print('Model did not return JSON. Raw output above.')
else:
    preds = json.loads(output[start:end])
    print(f'Parsed {len(preds)} predictions for {n} characters')

    img_w, img_h = img.size

    def iou(a, b):
        ax1 = a['x']
        ay1 = a['y']
        ax2 = a['x'] + a['w']
        ay2 = a['y'] + a['h']
        bx1 = b['x']
        by1 = b['y']
        bx2 = b['x'] + b['w']
        by2 = b['y'] + b['h']
        ix1 = max(ax1, bx1)
        iy1 = max(ay1, by1)
        ix2 = min(ax2, bx2)
        iy2 = min(ay2, by2)
        if ix2 <= ix1 or iy2 <= iy1:
            return 0.0
        inter = (ix2 - ix1) * (iy2 - iy1)
        union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
        return inter / union if union > 0 else 0.0

    scores = []
    for i, (pred, gt) in enumerate(zip(preds, boxes)):
        p = {
            'x': pred['x'] / 100 * img_w,
            'y': pred['y'] / 100 * img_h,
            'w': pred['w'] / 100 * img_w,
            'h': pred['h'] / 100 * img_h,
        }
        g = {'x': gt['x'], 'y': gt['y'], 'w': gt['w'], 'h': gt['h']}
        score = iou(p, g)
        scores.append(score)
        mark = '✓' if score >= 0.5 else ('~' if score >= 0.3 else '✗')
        print(f'[{i+1:2d}] {chars[i]}  IoU={score:.2f} {mark}')

    if scores:
        mean = sum(scores) / len(scores)
        good = sum(1 for s in scores if s >= 0.5)
        print(f'\nMean IoU: {mean:.2f}')
        print(f'Good (>=0.5): {good}/{len(scores)}')

In [ ]:
# Cell 7: Visualize — draw ground truth (green) vs predictions (blue)
import numpy as np
import cv2
from IPython.display import display

img_cv = cv2.imread('col_experiment.jpg')
vis = img_cv.copy()
font = cv2.FONT_HERSHEY_SIMPLEX

for b in boxes:
    x, y, w, h = int(b['x']), int(b['y']), int(b['w']), int(b['h'])
    cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 180, 0), 2)
    cv2.putText(vis, b['char'], (x+2, y-4), font, 0.6, (0, 180, 0), 1)

if start != -1:
    for pred in preds:
        x = int(pred['x'] / 100 * img_w)
        y = int(pred['y'] / 100 * img_h)
        w = int(pred['w'] / 100 * img_w)
        h = int(pred['h'] / 100 * img_h)
        cv2.rectangle(vis, (x, y), (x+w, y+h), (200, 60, 0), 2)
        cv2.putText(vis, pred.get('char', '?'), (x+2, y+h+14), font, 0.6, (200, 60, 0), 1)

vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
display(Image.fromarray(vis_rgb).resize((400, int(400 * vis_rgb.shape[0] / vis_rgb.shape[1]))))
print('Green = ground truth, Blue = model prediction')